# nb_validacion_bronze_sqlserver
| Campo      | Valor |
|------------|-------|
| Proyecto   | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa       | Bronze · validación (no productivo, no va en el job) |
| Propósito  | Verificar que la ingesta CDC llegó completa y trazable a Bronze |
| Pruebas    | 1. Filas por corrida (`_id_ejecucion`, `_op`) · 2. Carpetas en landing · 3. Recorrido de estados de pagos |
| Evidencia  | Simulador día 1 (`fuentes/logs/dia_20261012_003146.json`) · corrida `a59b5082` |
| Autoría    | Richard Huamán · con asistencia de IA (Claude) |
| Versión    | 1.0 · 2026-10-12 |

## 1. Filas por corrida en Bronze

In [0]:
%sql
-- =====================================================================
-- Diagnóstico · ¿llegaron a Bronze los cambios del día?
-- Agrupa payments_raw por corrida de extracción: si aparece un segundo
-- _id_ejecucion con ~319 filas (I/U) → los datos entraron (caso A).
-- Si solo aparece la carga inicial (R) → Auto Loader no los leyó (caso B).
-- =====================================================================
SELECT _id_ejecucion,
       _op,
       COUNT(*)            AS filas,
       MIN(_ingestion_ts)  AS primera_ingesta,
       MAX(_ingestion_ts)  AS ultima_ingesta
FROM andina_dev.bronze.payments_raw
GROUP BY ALL
ORDER BY primera_ingesta;

## 2. Carpetas de corridas en landing

In [0]:
# Diagnóstico · carpetas de corridas en landing para payments
# Espero 2 carpetas id_ejecucion=...: la carga inicial y la de hoy
for f in dbutils.fs.ls("/Volumes/andina_dev/landing/sqlserver/payments/"):
    print(f.path)
    for sub in dbutils.fs.ls(f.path):
        print("   ", sub.name)

## 3. Recorrido de estados de pagos (CDC)

In [0]:
%sql
-- =====================================================================
-- Verificación · historial de estados de pagos del simulador (día 1)
-- Esperado:
--   42213 → 3 filas: I pendiente → U aprobado → U reembolsado
--   42178 → U rechazado
--   42247 → reintento de 42178: I pendiente → U aprobado
-- =====================================================================
SELECT *
FROM andina_dev.bronze.payments_raw
WHERE PaymentID IN (42213, 42178, 42247)
ORDER BY PaymentID, _lsn, _seqval;   -- orden real en que ocurrieron los cambios